Linear Algebra

Calculus - Optimization

Activation Function

Loss Function

In [ ]:
import torch
import torch.nn as nn

# ===== Test MSE =====
y_true = torch.tensor([10.0,5.0,9.0])
y_pred = torch.tensor([8.0,7.0,6.0])

mse = nn.MSELoss()
print("MSE Loss:", mse(y_pred, y_true).item())

# ===== Test Cross Entropy =====
ce = nn.CrossEntropyLoss()

outputs = torch.tensor([[2.0, 0.5]])  # logits
labels = torch.tensor([0])           # true class index

print("Cross Entropy Loss:", ce(outputs, labels).item())   #transform logic to probality, calculate CE, .item() to change to number to print

Data Preprocessing

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA

data_main = {
   'CustomerID': [1, 2, 3, 4, 5],
   'Age': [25, 400, 22, 25, 45],
   'Income': [50000, 60000, -3, 50000, 80000]
}
df = pd.DataFrame(data_main)

print("Original Data")
print(df, "\n")

#Cleaning
df['Age'] = np.where((df['Age'] < 0) | (df['Age'] > 100), np.nan, df['Age'])
df['Income'] = np.where(df['Income'] < 0, np.nan, df['Income'])
df['Age'] = df['Age'].fillna(df['Age'].mean())
df['Income'] = df['Income'].fillna(df['Income'].median())

print("After cleaning")
print(df, "\n")

#Integration
data_sales = {
   'CustomerID': [1, 2, 3, 4, 5],
   'Country': ['VN', 'US', 'JP', 'VN', 'JP'],
   'Total_Spent': [100, 250, 300, 150, 400]
}
df_sales = pd.DataFrame(data_sales)
df = pd.merge(df, df_sales, on='CustomerID', how='inner')

print("After integration (merge with sales data):")
print(df, "\n")

#Discretization
bins = [0, 18, 35, 55, 120]
labels = ['Child', 'Young Adult', 'Middle-Aged', 'Senior']
df['Age_Group'] = pd.cut(df['Age'], bins=bins, labels=labels)

print("After discretization (Age -> Age_Group):")
print(df[['Age', 'Age_Group']], "\n")

#Transformation
df = pd.get_dummies(df, columns=['Country', 'Age_Group'], dtype=int)
scaler = MinMaxScaler()
df[['Income', 'Total_Spent']] = scaler.fit_transform(df[['Income', 'Total_Spent']])

print("After transformation (One-hot encoding và Min-Max Scaling):")
print(df, "\n")

#Reduction
features = df.drop(columns=['CustomerID', 'Age'])
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(features)
df_final = pd.DataFrame(reduced_data, columns=['PC1', 'PC2'])
df_final['CustomerID'] = df['CustomerID']

print("After PCA reduction")
print(df_final)


Autograd and Model Architecture

In [ ]:
#Autograd
import torch

# 1. Setup Data
x_train = torch.tensor([1.0, 2.0, 3.0])
y_true = torch.tensor([3.0, 5.0, 7.0]) 

# 2. Initialize Parameters (Starting Point)
w = torch.randn(1, requires_grad=True)
b = torch.randn(1, requires_grad=True)
learning_rate = 0.01

# 3. The Loop (Iterative Learning)
for epoch in range(100): # Running for 100 iterations
    # --- Step A: Forward Pass ---
    y_pred = w * x_train + b

    
    # --- Step B: Calculate Scalar Loss ---
    loss = torch.mean((y_pred - y_true)**2)
    
    # --- Step C: Backward Pass (Autograd) ---
    # This computes gradients for w and b
    loss.backward()
    
    # --- Step D: Optimization (Weight Update) ---
    with torch.no_grad():
        w -= learning_rate * w.grad
        b -= learning_rate * b.grad
        
        # IMPORTANT: Zero the gradients for the next iteration
        w.grad.zero_()
        b.grad.zero_()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}: Loss = {loss.item():.4f}, w = {w.item():.2f}, b = {b.item():.2f}")

#Model Architecture
import torch.nn as nn
import torch.nn.functional as F

class MNISTClassifier(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, output_dim=10):
        """
        Constructor: Defines the model architecture.
        """
        super().__init__() # Initializes the base nn.Module
        
        # Define Fully Connected (Linear) Layers
        self.fc1 = nn.Linear(input_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, output_dim)
        
        # Regularization layer
        self.dropout = nn.Dropout(0.2)

    def forward(self, x):
        """
        Forward Pass: Defines the computational graph.
        """
        # Step 1: Reshape 2D image (28x28) to 1D vector (784)
        x = x.view(x.size(0), -1) 
        
        # Step 2: Linear transformation followed by Non-linear activation (ReLU)
        x = F.relu(self.fc1(x))
        
        # Step 3: Apply Dropout to prevent overfitting
        x = self.dropout(x)
        
        # Step 4: Final linear layer to produce output Logits
        logits = self.fc2(x)
        return logits
# Customizing the architecture: input 784, hidden 256, output 10
model = MNISTClassifier(input_dim=784, hidden_dim=256, output_dim=10)
